In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

100%|██████████| 2.29G/2.29G [01:50<00:00, 22.3MB/s]

Extracting files...


Path to dataset files: C:\Users\alejo\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2


In [2]:
import shutil
import os

# 1. Descargar (o localizar si ya está descargado)
print("Localizando dataset...")
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print(f"Dataset encontrado en: {path}")

# 2. Definir tu carpeta de destino local
project_data_dir = os.path.join(os.getcwd(), "data")

# 3. Mover los archivos
# El dataset suele venir con una subcarpeta redundante 'chest_xray', vamos a limpiarlo.
source_folder = os.path.join(path, "chest_xray")

if os.path.exists(project_data_dir):
    print(f"¡Atención! La carpeta '{project_data_dir}' ya existe. Bórrala si quieres recargar los datos.")
else:
    print(f"Copiando archivos a {project_data_dir} (esto puede tardar unos segundos)...")
    try:
        # Copiamos todo el contenido a la carpeta data del proyecto
        shutil.copytree(source_folder, project_data_dir)
        print("✅ ¡Datos copiados exitosamente!")
        
        # Verificar estructura
        print("\nEstructura de carpetas creada:")
        for root, dirs, files in os.walk(project_data_dir):
            level = root.replace(project_data_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 4 * (level + 1)
            # Solo mostramos las primeras carpetas para no llenar la pantalla
            if level < 2: 
                pass
    except Exception as e:
        print(f"Error al copiar: {e}")
        print("Intenta copiar manualmente la carpeta 'chest_xray' del path original a tu carpeta 'data'.")

Localizando dataset...
Dataset encontrado en: C:\Users\alejo\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2
Copiando archivos a c:\Users\alejo\Desktop\FAMILIA\Alejo\TECH\developments\PERSONAL DEVELOPMENTS\Machine-learning-projects\computer-vision\ViT-Xray-Explainability\data (esto puede tardar unos segundos)...
✅ ¡Datos copiados exitosamente!

Estructura de carpetas creada:
data/
    chest_xray/
        test/
            NORMAL/
            PNEUMONIA/
        train/
            NORMAL/
            PNEUMONIA/
        val/
            NORMAL/
            PNEUMONIA/
    test/
        NORMAL/
        PNEUMONIA/
    train/
        NORMAL/
        PNEUMONIA/
    val/
        NORMAL/
        PNEUMONIA/
    __MACOSX/
        chest_xray/
            test/
                NORMAL/
                PNEUMONIA/
            train/
                NORMAL/
                PNEUMONIA/
            val/
                NORMAL/
                PNEUMONIA/


In [3]:
import torch
from torchvision import datasets, transforms
import os

# Definir ruta
data_dir = './data'
train_dir = os.path.join(data_dir, 'train')

# Transformaciones básicas (ViT necesita 224x224)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

try:
    # ImageFolder es la herramienta estándar para leer carpetas con nombres de clases
    train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
    
    print(f"✅ ¡Éxito! PyTorch pudo leer el dataset.")
    print(f"Imágenes de entrenamiento encontradas: {len(train_dataset)}")
    print(f"Clases detectadas: {train_dataset.classes}") # Debería decir ['NORMAL', 'PNEUMONIA']
    
    # Verificamos una imagen
    img, label = train_dataset[0]
    print(f"Forma del tensor de una imagen: {img.shape}") # Debería ser torch.Size([3, 224, 224])

except FileNotFoundError:
    print("❌ Error: No se encontró la carpeta 'data/train'. Asegúrate de haber corrido el paso 1.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

✅ ¡Éxito! PyTorch pudo leer el dataset.
Imágenes de entrenamiento encontradas: 5216
Clases detectadas: ['NORMAL', 'PNEUMONIA']
Forma del tensor de una imagen: torch.Size([3, 224, 224])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm # Nuestra librería de modelos SOTA
from tqdm import tqdm # Para la barra de progreso
import os

# --- CONFIGURACIÓN ---
# Si tienes tarjeta gráfica NVIDIA, usará 'cuda'. Si no, 'cpu' (será más lento)
DEVICE = "cpu" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16    # Cantidad de fotos que procesa a la vez (bájalo a 8 si te da error de memoria)
EPOCHS = 1         # Cuántas veces revisará todo el dataset (3 suele bastar para Fine-Tuning)
LEARNING_RATE = 2e-5 # Velocidad de aprendizaje (muy bajita para no romper lo que el modelo ya sabe)
DATA_DIR = './data'

def main():
    print(f"🚀 Iniciando entrenamiento en: {DEVICE}")

    # 1. PREPARAR LOS DATOS (Transformaciones)
    # Los modelos ViT suelen necesitar imágenes de 224x224
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(), # Data Augmentation (espejo) para que aprenda mejor
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
        'test': transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
    }

    # Cargar los datasets
    # Nota: Usamos la carpeta 'test' como validación porque la carpeta 'val' original es muy pequeña
    train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), data_transforms['train'])
    val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), data_transforms['test'])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"📸 Imágenes de entrenamiento: {len(train_dataset)}")
    print(f"📸 Imágenes de validación: {len(val_dataset)}")

    # 2. CARGAR EL MODELO (VISION TRANSFORMER)
    # 'vit_base_patch16_224': Es el estándar. 
    # num_classes=2: Lo forzamos a tener solo 2 salidas (Normal vs Neumonía)
    print("🧠 Descargando y cargando el modelo ViT...")
    model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
    model = model.to(DEVICE)

    # 3. CONFIGURAR MOTORES (Pérdida y Optimizador)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # 4. BUCLE DE ENTRENAMIENTO
    best_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
        
        # --- FASE DE ENTRENAMIENTO ---
        model.train()
        train_loss = 0
        train_correct = 0
        
        # Barra de progreso
        loop = tqdm(train_loader, desc="Entrenando")
        
        for images, labels in loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            # A. Forward (Predicción)
            outputs = model(images)
            loss = criterion(outputs, labels)

            # B. Backward (Aprendizaje)
            optimizer.zero_grad() # Limpiar basura anterior
            loss.backward()       # Calcular errores
            optimizer.step()      # Ajustar neuronas

            # Métricas
            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels.data)
            
            # Actualizar barra
            loop.set_postfix(loss=loss.item())

        train_acc = train_correct.double() / len(train_dataset)
        print(f"Resultados Train -> Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f}")

        # --- FASE DE VALIDACIÓN ---
        model.eval() # Modo evaluación (apaga el aprendizaje)
        val_correct = 0
        
        with torch.no_grad(): # No necesitamos calcular gradientes aquí (ahorra memoria)
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data)

        val_acc = val_correct.double() / len(val_dataset)
        print(f"Resultados Val   -> Acc: {val_acc:.4f}")

        # Guardar el mejor modelo
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "modelo_vit_neumonia.pth")
            print("💾 ¡Nuevo récord! Modelo guardado.")

    print("\n✅ ¡Entrenamiento finalizado!")

if __name__ == '__main__':
    main()

🚀 Iniciando entrenamiento en: cpu
📸 Imágenes de entrenamiento: 5216
📸 Imágenes de validación: 624
🧠 Descargando y cargando el modelo ViT...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

c:\Users\alejo\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alejo\.cache\huggingface\hub\models--timm--vit_base_patch16_224.augreg2_in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



--- Epoch 1/1 ---


Entrenando:  45%|████▌     | 148/326 [22:58<27:37,  9.31s/it, loss=0.0735] 


KeyboardInterrupt: 

delete all files when finish

In [ ]:
import shutil
import os

def eliminar_dataset(ruta_carpeta='./data'):
    # Verificamos si existe para no dar errores
    if os.path.exists(ruta_carpeta):
        try:
            print(f"🗑️ Eliminando dataset en '{ruta_carpeta}' para liberar espacio...")
            shutil.rmtree(ruta_carpeta) # Esto borra la carpeta y todo lo de adentro
            print("✅ ¡Limpieza completada! Espacio recuperado.")
        except Exception as e:
            print(f"❌ Error al eliminar: {e}")
    else:
        print("⚠️ La carpeta ya no existe o la ruta es incorrecta.")

if __name__ == "__main__":
    confirmacion = input("¿Estás seguro de borrar TODAS las imágenes en ./data? (s/n): ")
    if confirmacion.lower() == 's':
        eliminar_dataset()
    else:
        print("Operación cancelada.")